# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bitshank-2338/flyrank-ai-starter-notebook/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** prioritize a page for editorial review when it has at least 500 impressions in the trailing 90 days and has not been updated for at least 180 days; rank eligible pages by impressions. The single reason is `stale_visible`, and the action is `review_refresh`. This is a triage cue, not a diagnosis or an automatic edit.

The two descriptive checks below compare decline-proxy rates across staleness and exposure buckets, with counts. The label is used only to audit associations; the rule does not use it or its parent trend fields. Because the label and the input window are contemporaneous, these checks do not establish future prediction or causation.

In [1]:
import pandas as pd
from pathlib import Path
url = 'https://raw.githubusercontent.com/Bitshank-2338/flyrank-ai-starter-notebook/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
df['decline_proxy'] = df['trend_direction'].eq('down')
df['stale_bucket'] = pd.cut(df['days_since_last_update'], [-1, 89, 179, 364, float('inf')], labels=['0-89','90-179','180-364','365+'])
df['volume_bucket'] = pd.cut(df['impressions_90d'], [-1, 99, 499, 4999, float('inf')], labels=['0-99','100-499','500-4999','5000+'])
for col in ['stale_bucket','volume_bucket']:
    print(col)
    display(df.groupby(col, observed=True)['decline_proxy'].agg(n='size', decline_rate='mean'))
print('Overall decline proxy rate:', round(df.decline_proxy.mean(),3))

stale_bucket


,n,decline_rate
stale_bucket,,
0-89,20655,0.512031
90-179,9171,0.611057
180-364,169,0.467456
365+,5,0.600000


volume_bucket


,n,decline_rate
volume_bucket,,
0-99,7994,0.389042
100-499,5280,0.604356
500-4999,10575,0.623924
5000+,6151,0.546740


Overall decline proxy rate: 0.542


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
eligible = df['days_since_last_update'].ge(180) & df['impressions_90d'].ge(500)
queue = df.loc[eligible, ['client_id','content_id','days_since_last_update','impressions_90d','avg_position','ctr']].copy()
queue['score'] = queue['impressions_90d']
queue['reason_code'] = 'stale_visible'
queue['action'] = 'review_refresh'
queue = queue.sort_values(['score','content_id'], ascending=[False,True]).reset_index(drop=True)
Path('work/outputs').mkdir(parents=True, exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('Eligible items:', len(queue))
assert queue['score'].is_monotonic_decreasing
print('Queue saved locally; identifiers omitted from displayed review.')
display(queue.drop(columns=['client_id','content_id']).head(10))

Eligible items: 17
Queue saved locally; identifiers omitted from displayed review.


,days_since_last_update,impressions_90d,avg_position,ctr,score,reason_code,action
0,194,61678,19.7,0.15,61678,stale_visible,review_refresh
1,194,59472,24.8,0.13,59472,stale_visible,review_refresh
2,194,25715,22.2,0.23,25715,stale_visible,review_refresh
3,193,13299,10.5,0.49,13299,stale_visible,review_refresh
4,194,7812,39.0,0.01,7812,stale_visible,review_refresh
5,193,7558,17.9,0.20,7558,stale_visible,review_refresh
6,194,4590,31.0,0.00,4590,stale_visible,review_refresh
7,194,4556,16.4,0.33,4556,stale_visible,review_refresh
8,194,4429,25.3,0.38,4429,stale_visible,review_refresh
9,193,1697,15.8,0.12,1697,stale_visible,review_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.